In [1]:
import sys
sys.path.append('..')

In [2]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import ParameterGrid

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [3]:
# file_path = "../results/cost_validity/lr_synthetic_alg1_lamb0.1.pickle"
# ret = pd.read_pickle(file_path)

# x0 = ret['x_0'][1][0]
# x0_withBias = np.hstack((x0, np.array([1])))
# xR_old = ret['x_r'][1][0]
# xR_old_withBias = np.hstack((xR_old, np.array([1])))

# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) 
# divider = np.linalg.norm(theta0)
# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64) / divider
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) /divider
# theta0_withBias = np.hstack((theta0, bias0))

# alpha = 0.02


In [ ]:
alphas = (0.02, 0.04, 0.2)      # <------------- Don't Touch
clf = "lr"
datasets = ["sba"]
recourse_fns = [ROAR]    # <---------------------
lambdas = [0.1, 0.2, 0.3]       # <-------------
save_file = True



for dataset in datasets:
    # Read File and Get results
    file_path = f"../results/cost_validity_latest/{dataset}_correct.pickle"
    ret = pd.read_pickle(file_path)

    x0s = np.array(ret['x_0'])
    x0s = x0s[0]
    theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
    bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 

    ret['x_0'] = ret['x_0'][:len(alphas)]
    filtered_paramVal = [val['delta_max'] for val in ret['params'] if val['delta_max'] in alphas]
    ret['params'] = ParameterGrid({'delta_max': filtered_paramVal})

    for recourse_fn in recourse_fns:
        for lamb in lambdas:
            xRs = [] # Final recouuse list
            
            for alpha in alphas:
                res = []
                for x0 in tqdm.tqdm(x0s, desc=f"Running {clf}_{dataset} with lambda = {lamb}, alpha = {alpha}"):
                    reco = recourse_fn(weights=theta0, bias=bias0, alpha=alpha, lamb=lamb)
                    res.append(reco.get_recourse(x0))
                xRs.append(res)
            
            ret['x_r'] = xRs
            
            # Save each (dataset, recourse function, lambda) file
            if save_file:
                file_path_saved = f"../results/cost_validity_latest/{clf}_{dataset}_{reco.name}_lamb{lamb}_new.pickle"
                with open(file_path_saved, 'wb') as outFile:
                    pickle.dump(ret, outFile)

Running lr_sba with lambda = 0.3, alpha = 0.2: 100%|██████████| 100/100 [00:00<00:00, 4575.09it/s]
